# UBDS 2026: Basic Python — Day 5
## From beer aroma chemistry to statistics and machine learning

Today we use a real collection of **170 beer-associated aroma compounds** instead of the penguins dataset. Each row is a compound with a name, identifiers, and a SMILES representation. We calculate molecular features with RDKit and use them for visualisation, statistical testing, and machine learning.

### Learning goals

1. Briefly inspect a research table (pandas recap)
2. Turn SMILES strings into numerical RDKit descriptors
3. Explore chemical space with 2D/3D t-SNE and UMAP
4. Run and interpret a Welch t-test and one-way ANOVA
5. Train and evaluate a decision tree and random forest
6. Create an artificial beer mixture and predict a **chemical-family profile**

> **Important scientific limitation:** the supplied source contains compound identities and structures, but no measured concentrations or sensory-panel scores. Therefore, the final “flavour profile” is an educational, concentration-weighted profile of predicted chemical families—not a prediction of human taste or aroma intensity.

### How to use this notebook

Worked examples are followed by **🧪 Student task** cells. Each task cell contains valid starter code, so the notebook runs from top to bottom. Change the marked variables, re-run the cell, and write your interpretation in the markdown space provided. The goal is to justify your choices from the output.

---

**Українською:** Сьогодні ми працюємо зі справжнім набором сполук, пов'язаних з ароматом пива. Фінальний профіль є навчальним профілем хімічних класів, а не результатом сенсорної панелі.

## 1. Imports and reproducibility

The random seed makes stochastic plots and train/test splits repeatable. RDKit converts molecular structures into features; scikit-learn supplies embeddings and models.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.manifold import TSNE
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

try:
    import umap
except ImportError:
    umap = None

RANDOM_STATE = 2026
sns.set_theme(style="whitegrid", context="notebook")

## 2. Load and briefly inspect the data

Pandas was covered on Day 4, so we only check the shape, columns, first rows, and missingness. The notebook first looks for the portable course copy, then falls back to the original research directory.

In [ ]:
LOCAL_DATA = Path("data/beer_aroma_compounds.csv")
RESEARCH_DATA = Path(
    "/media/presteuser/0ef4bc7f-ad1a-455c-b8e0-a9983a65b1fd/"
    "home/80_polymers/beer_mixture_experiment/smiles_beer.csv"
)

data_path = LOCAL_DATA if LOCAL_DATA.exists() else RESEARCH_DATA
if not data_path.exists():
    raise FileNotFoundError(
        "Beer data not found. Run this notebook from day5/ or restore the research CSV."
    )

df = pd.read_csv(data_path, index_col=0)
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from {data_path}")
display(df.head(3))
display(df.isna().sum().rename("missing values").to_frame().T)

### A tiny inspection exercise

Print the column names and inspect five random compounds. Which columns describe molecular identity? Which would be unsuitable as a biological target?

In [ ]:
print(df.columns.tolist())
display(df.sample(5, random_state=RANDOM_STATE))

`pubchem_cid` is an identifier, not a meaningful continuous response. Predicting it with regression would be scientifically meaningless. We retain rows with valid molecular structures and calculate informative features instead.

### 🧪 Student task 1 — easy text filtering

Only change `search_word` first. Try `ethyl`, `acid`, `alcohol`, or `methyl`. Then complete the one-line scaffold below using the worked example as a pattern.

In [ ]:
# TODO: change the search word and, optionally, the selected columns
search_word = "ethyl"
columns_to_keep = ["input_name", "smiles", "pubchem_cid"]

name_subset = df.loc[
    df["input_name"].str.contains(search_word, case=False, na=False),
    columns_to_keep,
].copy()

print(f"Found {len(name_subset)} compounds containing {search_word!r}")
display(name_subset.head(10))

# YOUR TURN: uncomment and replace the blank.
# acid_compounds = df[df["input_name"].str.contains("_____", case=False, na=False)]
# display(acid_compounds.head())

**Interpretation (write 1–2 sentences):** What does one row represent? Did your search return what you expected, or did it also find surprising names?

## 3. SMILES → RDKit descriptors

A SMILES string is a compact text representation of molecular structure. The descriptors below measure size, polarity, flexibility, rings, and related chemical properties.

In [ ]:
df["mol"] = df["smiles"].apply(Chem.MolFromSmiles)
invalid = df["mol"].isna().sum()
df = df.loc[df["mol"].notna()].reset_index(drop=True)
print(f"Valid molecules: {len(df)}; invalid SMILES removed: {invalid}")

def rdkit_descriptors(mol):
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotBonds": Lipinski.NumRotatableBonds(mol),
        "RingCount": rdMolDescriptors.CalcNumRings(mol),
        "HeavyAtomCount": mol.GetNumHeavyAtoms(),
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),
    }

descriptor_df = pd.DataFrame([rdkit_descriptors(mol) for mol in df["mol"]])
feature_names = descriptor_df.columns.tolist()
beer = pd.concat([df.drop(columns="mol"), descriptor_df], axis=1)
display(beer[["input_name", "smiles"] + feature_names].head())

### Create one flavour-relevant chemical-family label

The source notebook classified overlapping functional groups. For supervised learning we need one target per row, so the function below uses a transparent priority rule. These labels are derived from structure; they are not measured sensory labels.

In [ ]:
FAMILY_PATTERNS = {
    "sulfur": Chem.MolFromSmarts("[S]"),
    "acid": Chem.MolFromSmarts("C(=O)[OH]"),
    "ester/lactone": Chem.MolFromSmarts("C(=O)O[#6]"),
    "phenolic/aromatic": Chem.MolFromSmarts("a"),
    "alcohol": Chem.MolFromSmarts("[OX2H;!$(OC=O)]"),
    "carbonyl": Chem.MolFromSmarts("[CX3]=[OX1]"),
}

def chemical_family(mol):
    for family, pattern in FAMILY_PATTERNS.items():
        if mol.HasSubstructMatch(pattern):
            return family
    return "other"

beer["chemical_family"] = df["mol"].apply(chemical_family)
family_counts = beer["chemical_family"].value_counts()
display(family_counts.rename("n").to_frame())

### 🧪 Student task 2 — filtering, one small step at a time

Run each example, then change only the value marked `TODO`. These patterns cover equality, numerical comparison, two conditions, and sorting.

In [ ]:
# A. Equality
family_to_keep = "alcohol"  # TODO: try "sulfur" or "carbonyl"
one_family = beer[beer["chemical_family"] == family_to_keep]
display(one_family[["input_name", "chemical_family", "MolWt"]].head())

# B. Numerical comparison
weight_limit = 150  # TODO: try 100 or 200
heavy_compounds = beer[beer["MolWt"] > weight_limit]
print("Compounds above the limit:", len(heavy_compounds))

# C. Two conditions
family_choice = "ester/lactone"  # TODO
logp_limit = 2.0                 # TODO
two_conditions = beer[
    (beer["chemical_family"] == family_choice)
    & (beer["LogP"] > logp_limit)
]
display(two_conditions[["input_name", "chemical_family", "LogP"]].head())

# D. Sort and show five rows
sort_column = "TPSA"       # TODO: use another feature name
ascending_order = False    # TODO: switch to True
sorted_compounds = beer.sort_values(sort_column, ascending=ascending_order)
display(sorted_compounds[["input_name", sort_column]].head(5))

**Fill-in scaffolds:** copy these into a new cell and replace the blanks.

```python
low_tpsa = beer[beer["_____"] < _____]
small_alcohols = beer[(beer["chemical_family"] == "_____") & (beer["_____"] < _____)]
two_families = beer[(beer["chemical_family"] == "_____") | (beer["chemical_family"] == "_____")]
```

**Interpretation:** What is the difference between `&` and `|`?

### 🧪 Student task 3 — group and summarise

Change one function or descriptor at a time.

In [ ]:
counts_by_family = beer["chemical_family"].value_counts(ascending=False)
display(counts_by_family)

# TODO: replace MolWt with LogP or TPSA; replace mean() with median(), min(), or max()
mean_weight_by_family = beer.groupby("chemical_family")["MolWt"].mean()
display(mean_weight_by_family.sort_values())

# Scaffold to complete in a new cell:
# summary = beer.groupby("_____")["_____"]._____()

In [ ]:
plt.figure(figsize=(9, 4))
sns.countplot(
    data=beer,
    y="chemical_family",
    order=family_counts.index,
    hue="chemical_family",
    palette="tab10",
    legend=False,
)
plt.title("Beer-associated compounds by chemical family")
plt.xlabel("Number of compounds")
plt.ylabel("")
plt.tight_layout()
plt.show()

### 🧪 Student task 4 — three easy plot patterns

Start by changing only the variable names. After each plot, add a title and axis labels of your own.

In [ ]:
# A. Histogram: distribution of one variable
histogram_variable = "MolWt"  # TODO: try LogP or TPSA
plt.figure(figsize=(8, 4))
sns.histplot(data=beer, x=histogram_variable, bins=20, kde=True)
plt.title(f"Distribution of {histogram_variable}")
plt.show()

# B. Box plot: compare one variable among groups
boxplot_variable = "TPSA"  # TODO: try MolWt or LogP
plt.figure(figsize=(10, 4))
sns.boxplot(data=beer, x="chemical_family", y=boxplot_variable)
plt.xticks(rotation=35, ha="right")
plt.title(f"{boxplot_variable} by chemical family")
plt.tight_layout()
plt.show()

# C. Scatter plot: relationship between two variables
x_descriptor = "MolWt"
y_descriptor = "LogP"  # TODO: change one or both

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=beer, x=x_descriptor, y=y_descriptor,
    hue="chemical_family", alpha=0.75, palette="tab10",
)
plt.title(f"{y_descriptor} versus {x_descriptor}")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

**Plot interpretation (write 2–3 sentences):** Is the relationship positive, negative, nonlinear, or absent? Which families overlap, and are there possible outliers? Does this plot establish causation?

**Fill-in scaffolds:**

```python
sns.histplot(data=beer, x="_____", bins=_____)
sns.boxplot(data=beer, x="chemical_family", y="_____")
sns.scatterplot(data=beer, x="_____", y="_____", hue="chemical_family")
```

## 4. Explore chemical space with dimensionality reduction

Nine descriptors are still difficult to view at once. We standardise them, then compress them to two or three coordinates.

- **t-SNE** emphasises local neighbourhoods; distances between far-away clusters should not be over-interpreted.
- **UMAP** also preserves local structure and often retains more global organisation.
- Both axes are synthetic and have no direct chemical units.

In [ ]:
X = beer[feature_names]
y = beer["chemical_family"]
X_scaled = StandardScaler().fit_transform(X)

n_samples = len(beer)
perplexity = min(25, max(5, (n_samples - 1) // 4))

tsne_2d = TSNE(
    n_components=2,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE,
).fit_transform(X_scaled)

if umap is not None:
    umap_2d = umap.UMAP(
        n_components=2, n_neighbors=15, min_dist=0.15,
        random_state=RANDOM_STATE,
    ).fit_transform(X_scaled)
else:
    warnings.warn("umap-learn is absent; PCA is shown as a fallback.")
    umap_2d = PCA(n_components=2).fit_transform(X_scaled)

In [ ]:
def plot_embedding_2d(ax, embedding, title):
    plot_df = pd.DataFrame({
        "axis 1": embedding[:, 0],
        "axis 2": embedding[:, 1],
        "family": y.to_numpy(),
    })
    sns.scatterplot(
        data=plot_df, x="axis 1", y="axis 2", hue="family",
        palette="tab10", s=65, alpha=0.8, ax=ax,
    )
    ax.set_title(title)
    ax.legend(title="Family", bbox_to_anchor=(1.02, 1), loc="upper left")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_embedding_2d(axes[0], tsne_2d, "2D t-SNE of RDKit descriptors")
plot_embedding_2d(
    axes[1], umap_2d,
    "2D UMAP of RDKit descriptors" if umap is not None else "2D PCA fallback",
)
plt.tight_layout()
plt.show()

### 3D embedding

Run the following cell and drag the figure to rotate it. A 3D view can reveal overlaps hidden in 2D, but it does not make the embedding more “true.”

In [ ]:
tsne_3d = TSNE(
    n_components=3,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    random_state=RANDOM_STATE,
).fit_transform(X_scaled)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
for family in family_counts.index:
    mask = y.eq(family).to_numpy()
    ax.scatter(
        tsne_3d[mask, 0], tsne_3d[mask, 1], tsne_3d[mask, 2],
        label=family, s=45, alpha=0.8,
    )
ax.set(title="3D t-SNE of beer aroma-compound chemical space",
       xlabel="t-SNE 1", ylabel="t-SNE 2", zlabel="t-SNE 3")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 🧪 Student task 5 — test a t-SNE parameter

Change `perplexity` to 5 and then 40. Do the visible clusters remain stable? Why should t-SNE not be used as proof that discrete natural groups exist?

In [ ]:
# 🧪 TODO: run this cell with values such as 5, 15, 25, and 40
student_perplexity = 15

if not 1 <= student_perplexity < len(beer):
    raise ValueError("perplexity must be positive and smaller than the sample size")

student_tsne = TSNE(
    n_components=2, perplexity=student_perplexity,
    init="pca", learning_rate="auto", random_state=RANDOM_STATE,
).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(9, 6))
plot_embedding_2d(ax, student_tsne, f"t-SNE, perplexity={student_perplexity}")
plt.tight_layout()
plt.show()

**Interpretation:** Record which structures remain visually close across parameter values and which clusters move or split. Distances and axis directions from separate t-SNE runs cannot be compared numerically.

**Optional UMAP task:** change `n_neighbors` and `min_dist` in the UMAP cell. Predict what each parameter will change before running it.

## 5. Statistical tests

Here statistical tests answer narrow questions about descriptor distributions. The observations are different compounds, not replicated beer measurements, so inference applies only to this compound collection.

### Welch t-test: do ester/lactone compounds have different LogP?

- **H₀:** mean LogP is equal for ester/lactone and non-ester compounds.
- **H₁:** the means differ.

Welch's test does not require equal group variances.

In [ ]:
ester_logp = beer.loc[beer["chemical_family"].eq("ester/lactone"), "LogP"]
other_logp = beer.loc[~beer["chemical_family"].eq("ester/lactone"), "LogP"]

t_result = stats.ttest_ind(ester_logp, other_logp, equal_var=False)
mean_difference = ester_logp.mean() - other_logp.mean()

print(f"Ester/lactone n={len(ester_logp)}, mean LogP={ester_logp.mean():.2f}")
print(f"All other n={len(other_logp)}, mean LogP={other_logp.mean():.2f}")
print(f"Mean difference={mean_difference:.2f}")
print(f"Welch t={t_result.statistic:.2f}, p={t_result.pvalue:.4g}")
print("Reject H0" if t_result.pvalue < 0.05 else "Do not reject H0")

In [ ]:
plt.figure(figsize=(8, 4))
test_plot = beer.assign(
    ester_group=np.where(
        beer["chemical_family"].eq("ester/lactone"),
        "ester/lactone", "all other compounds"
    )
)
sns.boxplot(data=test_plot, x="ester_group", y="LogP", hue="ester_group", legend=False)
sns.stripplot(data=test_plot, x="ester_group", y="LogP", color="black", alpha=0.4)
plt.title("LogP comparison used in the Welch t-test")
plt.xlabel("")
plt.tight_layout()
plt.show()

### 🧪 Student task 6 — repeat the t-test with another variable

This is the same test pattern as above, but the family and descriptor are variables. Change only those two values. Use a family with several observations.

In [ ]:
test_family = "alcohol"  # TODO: try "phenolic/aromatic" or "carbonyl"
test_descriptor = "TPSA"  # TODO: try "MolWt" or "LogP"

group_1 = beer.loc[beer["chemical_family"] == test_family, test_descriptor]
group_2 = beer.loc[beer["chemical_family"] != test_family, test_descriptor]
student_t_test = stats.ttest_ind(group_1, group_2, equal_var=False)

print(f"{test_family}: n={len(group_1)}, mean={group_1.mean():.2f}")
print(f"all others: n={len(group_2)}, mean={group_2.mean():.2f}")
print(f"t={student_t_test.statistic:.2f}, p={student_t_test.pvalue:.4g}")

# Fill-in pattern:
# new_group = beer.loc[beer["chemical_family"] == "_____", "_____"]

**Interpretation scaffold:** “We [reject / do not reject] H₀ because p is [smaller / larger] than 0.05. The data provide [evidence / insufficient evidence] that the mean _____ differs between _____ and the other compounds.”

### One-way ANOVA: does molecular weight differ among common families?

ANOVA tests whether all selected group means are equal. We use the four largest families so every group has a useful sample size. If ANOVA is significant, Tukey HSD explores which pairs differ.

In [ ]:
anova_families = family_counts.head(4).index.tolist()
anova_data = beer.loc[
    beer["chemical_family"].isin(anova_families),
    ["chemical_family", "MolWt"],
].copy()
groups = [
    group["MolWt"].to_numpy()
    for _, group in anova_data.groupby("chemical_family", sort=False)
]

anova_result = stats.f_oneway(*groups)
display(anova_data.groupby("chemical_family")["MolWt"].agg(["count", "mean", "std"]))
print(f"F={anova_result.statistic:.2f}, p={anova_result.pvalue:.4g}")

if anova_result.pvalue < 0.05:
    print(pairwise_tukeyhsd(
        endog=anova_data["MolWt"],
        groups=anova_data["chemical_family"],
        alpha=0.05,
    ))

### Assumptions and interpretation

Classical t-tests/ANOVA assume independent observations and approximately normal residuals; ANOVA also assumes similar variances. With small or strongly skewed groups, consider Mann–Whitney U or Kruskal–Wallis. A small p-value is not an effect size and does not imply sensory importance.

**Exercise:** replace `MolWt` with `TPSA`. State H₀, run ANOVA, and interpret the result in one sentence.

In [ ]:
# 🧪 Student task 7 — test another variable with ANOVA
# TODO: select another descriptor from feature_names
anova_descriptor = "TPSA"

student_anova_data = beer.loc[
    beer["chemical_family"].isin(anova_families),
    ["chemical_family", anova_descriptor],
].copy()
student_groups = [
    group[anova_descriptor].to_numpy()
    for _, group in student_anova_data.groupby("chemical_family", sort=False)
]
student_anova = stats.f_oneway(*student_groups)

display(student_anova_data.groupby("chemical_family")[anova_descriptor].agg(["count", "mean", "std"]))
print(f"ANOVA for {anova_descriptor}: F={student_anova.statistic:.2f}, p={student_anova.pvalue:.4g}")

**Your statistical conclusion:** Write H₀, report the p-value, make a reject/do-not-reject decision at α = 0.05, and explain what the result does *not* tell us. If significant, a post-hoc test is needed to identify differing pairs.

## 6. A fun supervised-learning model

We predict the structure-derived chemical family using RDKit descriptors. Because some descriptors helped define the labels, this is a demonstration of a workflow—not independent discovery. We stratify the split so class proportions remain similar.

In [ ]:
model_data = beer.loc[beer["chemical_family"].map(family_counts) >= 5].copy()
X_model = model_data[feature_names]
y_model = model_data["chemical_family"]

X_train, X_test, y_train, y_test = train_test_split(
    X_model, y_model,
    test_size=0.25,
    stratify=y_model,
    random_state=RANDOM_STATE,
)
print("Training rows:", len(X_train), "Test rows:", len(X_test))
print("Classes:", sorted(y_model.unique()))

### Decision tree: easy to explain

A shallow tree produces human-readable rules but may miss complex boundaries.

In [ ]:
tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
tree.fit(X_train, y_train)
tree_pred = tree.predict(X_test)
print(f"Decision-tree test accuracy: {accuracy_score(y_test, tree_pred):.3f}")

plt.figure(figsize=(20, 9))
plot_tree(
    tree,
    feature_names=feature_names,
    class_names=tree.classes_,
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("How the decision tree separates chemical families")
plt.show()

### Random forest: many diverse trees vote together

A random forest is harder to draw but usually more stable. We report held-out metrics and stratified cross-validation rather than training accuracy alone.

In [ ]:
forest = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
forest.fit(X_train, y_train)
forest_pred = forest.predict(X_test)

print(f"Random-forest test accuracy: {accuracy_score(y_test, forest_pred):.3f}\n")
print(classification_report(y_test, forest_pred, zero_division=0))

fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay.from_predictions(
    y_test, forest_pred, normalize="true", cmap="Blues",
    xticks_rotation=45, ax=ax,
)
ax.set_title("Random-forest confusion matrix (row-normalised)")
plt.tight_layout()
plt.show()

### 🧪 Student task 8 — tune model parameters

Change one parameter at a time and compare both training and test accuracy. A large training–test gap is evidence of overfitting; test accuracy from one split is still uncertain.

In [ ]:
# TODO: try max_depth = 2, 4, 8, None and min_samples_leaf = 1, 3, 8
student_max_depth = 4
student_min_samples_leaf = 3

student_tree = DecisionTreeClassifier(
    max_depth=student_max_depth,
    min_samples_leaf=student_min_samples_leaf,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
student_tree.fit(X_train, y_train)

student_results = pd.Series({
    "max_depth": student_max_depth,
    "min_samples_leaf": student_min_samples_leaf,
    "training_accuracy": student_tree.score(X_train, y_train),
    "test_accuracy": student_tree.score(X_test, y_test),
})
display(student_results.to_frame("value"))

**Model interpretation:** Which setting underfits? Which appears to overfit? Choose a final setting and justify it using the train/test gap, not test accuracy alone.

The next cell repeats the same experiment automatically.

In [ ]:
# Change this list to test fewer or more depths.
depths_to_test = [1, 2, 3, 4, 6, 10, None]
depth_results = []

for depth in depths_to_test:
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    depth_results.append({
        "max_depth": str(depth),
        "training_accuracy": model.score(X_train, y_train),
        "test_accuracy": model.score(X_test, y_test),
    })

depth_results = pd.DataFrame(depth_results)
display(depth_results)

depth_results.plot(
    x="max_depth", y=["training_accuracy", "test_accuracy"],
    marker="o", figsize=(8, 4), ylim=(0, 1.05),
)
plt.ylabel("Accuracy")
plt.title("Decision-tree depth: underfitting and overfitting")
plt.show()

### 🧪 Student task 9 — change random-forest parameters

Change one value at a time. More trees usually make predictions more stable but take longer; larger leaves make the model simpler.

In [ ]:
student_n_trees = 100       # TODO: try 20, 100, or 500
student_leaf_size = 2       # TODO: try 1, 3, or 8
student_max_features = "sqrt"  # TODO: try None or 0.5

student_forest = RandomForestClassifier(
    n_estimators=student_n_trees,
    min_samples_leaf=student_leaf_size,
    max_features=student_max_features,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
student_forest.fit(X_train, y_train)
print("Training accuracy:", round(student_forest.score(X_train, y_train), 3))
print("Test accuracy:    ", round(student_forest.score(X_test, y_test), 3))

# Scaffold:
# another_forest = RandomForestClassifier(n_estimators=_____, min_samples_leaf=_____, random_state=RANDOM_STATE)

In [ ]:
smallest_class = y_model.value_counts().min()
n_splits = min(5, int(smallest_class))
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(clone(forest), X_model, y_model, cv=cv, scoring="balanced_accuracy")
print("Balanced-accuracy CV scores:", np.round(cv_scores, 3))
print(f"Mean ± SD: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

### Which descriptors matter to the fitted forest?

Permutation importance measures how much test accuracy falls when one feature is shuffled. It is model- and dataset-specific and correlated descriptors may share importance.

In [ ]:
importance = permutation_importance(
    forest, X_test, y_test,
    n_repeats=30,
    scoring="balanced_accuracy",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importance.importances_mean,
    "sd": importance.importances_std,
}).sort_values("importance", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(importance_df["feature"], importance_df["importance"],
         xerr=importance_df["sd"])
plt.xlabel("Decrease in balanced accuracy after shuffling")
plt.title("Test-set permutation importance")
plt.tight_layout()
plt.show()

## 7. Build an artificial beer and predict its profile

We choose real compounds from the source table and assign hypothetical concentrations in mg/L. The classifier predicts a probability distribution for every compound. We then weight those probabilities by concentration and sum them.

This is a simple mixture calculation: it does **not** account for odour thresholds, nonlinear interactions, volatility, matrix effects, or human perception.

In [ ]:
# Explore valid names before constructing a mixture.
# TODO: change the search term to find compounds you might include.
compound_search = "ethyl"
display(
    beer.loc[
        beer["input_name"].str.contains(compound_search, case=False, na=False),
        ["input_name", "chemical_family"],
    ].sort_values("input_name").head(25)
)

In [ ]:
artificial_beer = pd.DataFrame({
    "input_name": [
        "Isoamyl acetate",     # commonly associated with banana-like aroma
        "Ethyl hexanoate",     # fruity ester
        "Linalool",            # hop-associated terpene alcohol
        "4-vinylguaiacol",     # phenolic compound
        "Diacetyl",            # buttery-associated carbonyl
        "Dimethyl sulfide",    # sulfur compound
    ],
    "concentration_mg_L": [2.2, 0.35, 0.08, 0.45, 0.12, 0.04],
})

mixture = artificial_beer.merge(
    beer[["input_name", "smiles"] + feature_names],
    on="input_name",
    how="left",
    validate="one_to_one",
)
if mixture[feature_names].isna().any().any():
    raise ValueError("At least one artificial-beer compound was not found.")

compound_probabilities = forest.predict_proba(mixture[feature_names])
probability_df = pd.DataFrame(compound_probabilities, columns=forest.classes_)
mixture_predictions = pd.concat(
    [mixture[["input_name", "concentration_mg_L"]].reset_index(drop=True), probability_df],
    axis=1,
)
mixture_predictions["predicted_family"] = forest.predict(mixture[feature_names])
display(mixture_predictions)

In [ ]:
weights = mixture["concentration_mg_L"].to_numpy()
profile = pd.Series(
    np.average(compound_probabilities, axis=0, weights=weights),
    index=forest.classes_,
    name="weighted profile",
).sort_values(ascending=False)

display((100 * profile).round(1).rename("profile_percent").to_frame())

plt.figure(figsize=(9, 4))
sns.barplot(x=profile.index, y=100 * profile.values, hue=profile.index,
            palette="tab10", legend=False)
plt.ylabel("Concentration-weighted predicted probability (%)")
plt.xlabel("Predicted chemical family")
plt.title("Artificial beer: educational chemical-family profile")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

### 🧪 Student task 10 — generate your own artificial sample

Edit `my_compounds` and `my_concentrations`. Names must exactly match `beer["input_name"]`, and concentrations must be non-negative. The helper gives a useful error if a name is unknown.

In [ ]:
def predict_mixture_profile(compound_names, concentrations_mg_L, fitted_model=forest):
    if len(compound_names) != len(concentrations_mg_L):
        raise ValueError("Provide one concentration for every compound name.")
    if len(compound_names) == 0:
        raise ValueError("The mixture must contain at least one compound.")
    concentrations_mg_L = np.asarray(concentrations_mg_L, dtype=float)
    if np.any(concentrations_mg_L < 0) or concentrations_mg_L.sum() <= 0:
        raise ValueError("Concentrations must be non-negative and sum to more than zero.")

    recipe = pd.DataFrame({
        "input_name": compound_names,
        "concentration_mg_L": concentrations_mg_L,
    })
    recipe_features = recipe.merge(
        beer[["input_name"] + feature_names], on="input_name", how="left"
    )
    missing_names = recipe_features.loc[
        recipe_features[feature_names].isna().any(axis=1), "input_name"
    ].tolist()
    if missing_names:
        raise ValueError(f"Unknown compound name(s): {missing_names}")

    probabilities = fitted_model.predict_proba(recipe_features[feature_names])
    predicted_profile = pd.Series(
        np.average(probabilities, axis=0, weights=recipe_features["concentration_mg_L"]),
        index=fitted_model.classes_, name="profile_fraction",
    ).sort_values(ascending=False)
    return recipe_features[["input_name", "concentration_mg_L"]], predicted_profile


# TODO: replace names and concentrations with your own recipe
my_compounds = ["Isoamyl acetate", "Linalool", "Diacetyl"]
my_concentrations = [1.5, 0.12, 0.08]

my_recipe, my_profile = predict_mixture_profile(my_compounds, my_concentrations)
display(my_recipe)
display((100 * my_profile).round(1).rename("profile_percent").to_frame())

plt.figure(figsize=(9, 4))
sns.barplot(x=my_profile.index, y=100 * my_profile.values, hue=my_profile.index,
            palette="tab10", legend=False)
plt.ylabel("Concentration-weighted predicted probability (%)")
plt.xlabel("Predicted chemical family")
plt.title("My artificial sample")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

**Mixture interpretation:** Which compound dominates by concentration? Does it also dominate the predicted profile? Change one concentration by a factor of ten and explain the result. Finally, explain why this chart must not be called a sensory flavour prediction.

### Final challenge

1. Change the concentrations or replace compounds with other names from `beer["input_name"]`.
2. Re-run the last two cells. How does the profile change?
3. Add an `odour_threshold_mg_L` column and calculate concentration/threshold before weighting. Why might this be closer to sensory relevance?
4. For a real sensory predictor, what new table would we need? Think about rows, predictors, targets, replication, and train/test splitting by beer rather than by molecule.

## Take-home messages

- Molecular strings can be transformed into numerical features.
- t-SNE and UMAP are exploratory maps, not statistical tests.
- A t-test compares two means; ANOVA compares several means.
- Evaluation must use unseen data and should reflect class imbalance.
- A model can only learn the target present in its training data. Chemical-family probabilities must not be presented as measured human flavour perception.